# 06. History injection and foundation models

This notebook rebuilds, from the study's saved results:

- **Table S-XII** (`reports/table10_history.csv`): history injection, ASR after $k$ = 1, 2, 4, 8 injected same-card
  transactions;
- **Table V** (`reports/tables/foundation_models.csv`): the foundation models beside the MLP and the HGNN;
- **Table S-XXVIII** (`reports/table17_fm_adapt.csv`): LoRA and zero-shot foundation models (only when
  `FM_ADAPT_RUNS` points at their runs);
- **Table S-XLIII** (`reports/table12_groups.csv`): IEEE-CIS clients with and without an identifier.

Inputs: `runs/<dataset>/attacks/history*.csv` and `history_per_example_pe.npz` (fraudadv.attacks.history), the attack
grid `runs/<dataset>/attacks/results*.csv` and `per_example*.npz`, `runs/<dataset>/<family>/metrics.json`,
`test_scores_seed0.npy` and `ablation_noemb.json`, `data/processed/ieee_cis/`; for Table S-XXVIII the LoRA / zero-shot
runs (`FM_ADAPT_RUNS`) and the RL attackers' results (`FRAUDADV_RL_RESULTS`). Table S-XII is computed first because
Table V reads the history table.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

The output folders, created up front as `scripts/make_results.sh` does (`mkdir -p reports/tables`).

In [ ]:
(ROOT / "reports" / "tables").mkdir(parents=True, exist_ok=True)

## Table S-XII: history injection

`make_report.history_table` pivots every dataset's `history.csv` and `history_fm_*.csv` into
`reports/table10_history.csv` (ASR and dollar-weighted ASR per $k$, for the no-query "mimic" attacker and the best of
four amounts); Table S-XII is the best-of-four rows. Table V reads it.

In [ ]:
from fraudadv.report import make_report as MR
from fraudadv.report import tables

MR.history_table()
H = nb.show_csv("reports/table10_history.csv", rows=12)

## Table V: foundation models

`tables.foundation_models` puts Chronos-2 and TimesFM 2.5 beside the MLP and the HGNN on the four datasets with card
or account sequences: clean AUC and AUPRC (means over three seeds), AUPRC with the encoding set to zero
(`ablation_noemb.json`), worst-case ASR at $\varepsilon = 0.25$ from the attack grid, and history injection with eight
transactions (from the table above), into `reports/tables/foundation_models.csv`.

In [ ]:
A = MR.load_attacks()          # every row of the attack grid
tables.foundation_models(A).round(3)

## Table S-XXVIII: LoRA and zero-shot foundation models

`fraudadv.report.fm_adapt` compares the frozen foundation models with LoRA-adapted and zero-shot versions on Sparkov
and TabFormer: clean metrics, worst-case ASR, the RL memory attacker, history injection and robust recall. It writes
`reports/table17_fm_adapt.csv` and `reports/pareto_extra_fm_adapt.csv` (points of Figure S7), as
`scripts/make_results.sh` runs it (`--adapt-runs "$FM_ADAPT_RUNS"`). The LoRA / zero-shot runs are not part of
`runs/`; without `FM_ADAPT_RUNS` this step is skipped.

In [ ]:
import os

FM_ADAPT_RUNS = os.environ.get("FM_ADAPT_RUNS")
if FM_ADAPT_RUNS:
    from fraudadv.report import fm_adapt

    T = fm_adapt.main(MR.RUNS, fm_adapt.RL_ROOT / "results_adapt", MR.REP, Path(FM_ADAPT_RUNS))
else:
    print("FM_ADAPT_RUNS is not set: Table S-XXVIII skipped. Point it at the LoRA / zero-shot runs to build it.")

## Table S-XLIII: clients with and without an identifier

`make_report.groups_table` splits IEEE-CIS by whether a transaction has a client identifier (no billing address, no
id): clean AUC and recall at the deployed threshold per group, worst-case ASR at $\varepsilon = 0.25$ from the
per-example grid outcomes, and history injection (best of four, $k = 8$) from `history_per_example_pe.npz`. It writes
`reports/table12_groups.csv` (also the source of the history-injection numbers of Section VI-E).

In [ ]:
MR.groups_table()
nb.show_csv("reports/table12_groups.csv")

## Run it yourself: history injection on a few Sparkov frauds

History injection with $k = 1$ against the saved XGBoost (seed 0) on the first 40 frauds of the Sparkov attack
sample, in memory only. The cell uses the pieces `fraudadv.attacks.history.run` is built from: what the attacker
knows about each fraud and the four candidate amounts (`_attacker_inputs`), the injected scenario (`scenarios`,
delayed by `timeline` when the card's last transaction is too recent), and the scoring (`_scenario_logits`). As in
`run`, a scenario fails when the injected transaction is itself flagged, and the attacker keeps the lowest-scoring of
the four amounts. The outcomes are compared fraud by fraud with the stored ones (`history_per_example_pe.npz`).
The ASR on 40 frauds is an illustration on a small sample, not the study's number (Table S-XII uses all frauds of the
sample). `N_LIVE = None` attacks the whole sample (228 frauds, a few seconds) and gives the stored $k = 1$ row of
`history.csv`.

In [ ]:
import json

import numpy as np
import pandas as pd

from fraudadv.attacks import history as HI
from fraudadv.attacks.run_attacks import RUNS, context_ids, load_model
from fraudadv.data.load import load

DS_LIVE, FAM_LIVE, K, N_LIVE = "sparkov", "xgboost", 1, 40
a, meta = load(DS_LIVE)
out = RUNS / DS_LIVE / "attacks"
tau = float(json.loads((out / "tau.json").read_text())[FAM_LIVE])        # deployed threshold (logit scale)
S = np.load(out / "sample.npz")["S"][:N_LIVE]                             # the first N_LIVE frauds of the attack sample
ids, X0, amt, t0, inj, dt_prev, tmpl, cands = HI._attacker_inputs(DS_LIVE, a, meta, S, context_ids(DS_LIVE))
m = load_model(DS_LIVE, FAM_LIVE, 0)
l0 = m.logit(X0, ids)
detected = l0 > tau
delay = HI.timeline(dt_prev, K)[0]
L = {}
for c, amounts in cands.items():                                          # the four candidate injected amounts
    Xt, Xi = HI.scenarios(inj, X0, t0, tmpl, amounts, K, dt_prev)
    lt, li = HI._scenario_logits(m, FAM_LIVE, DS_LIVE, Xt, Xi, ids, K, delay)
    L[c] = np.where(li < tau, lt, np.maximum(lt, l0))                     # a flagged injection makes the scenario fail
best = np.min(np.stack(list(L.values())), 0)
live = {"mimic (no queries)": detected & (L["mean_prev"] < tau), "best of 4 (4 queries)": detected & (best < tau)}

PE = np.load(out / "history_per_example_pe.npz")
assert np.array_equal(PE["S"][:N_LIVE], S)
print(f"{DS_LIVE} {FAM_LIVE}, k = {K}, {len(S)} frauds: {int(detected.sum())} detected "
      f"(stored: {int(PE[f'{FAM_LIVE}|detected'][:N_LIVE].sum())})")
for variant, evaded in live.items():
    stored = PE[f"{FAM_LIVE}|{variant}|{K}"][:N_LIVE]
    print(f"  {variant:<22} live ASR {evaded.sum() / max(detected.sum(), 1):.3f}, "
          f"same outcome as stored on {int((evaded == stored).sum())}/{len(S)} frauds")
stored_row = pd.read_csv(out / "history.csv").query("model == @FAM_LIVE and k == @K")
print("stored k = 1 rows on the whole sample (history.csv):")
display(stored_row[["attack", "k", "n_detected", "asr", "mean_logit_drop"]])
pd.DataFrame({"clean logit - tau": l0 - tau, "logit after injection (best amount) - tau": best - tau, "detected": detected,
              "evaded (live)": live["best of 4 (4 queries)"],
              "evaded (stored)": PE[f"{FAM_LIVE}|best of 4 (4 queries)|{K}"][:N_LIVE]}).round(3).head(10)

## Rerun the runs behind these results from scratch

The calls `scripts/reproduce.sh` makes for history injection (stage `attacks`) and for the consistent attacker's
history injection (stage `consistent`, Table S-XIX), for one dataset. History injection needs card histories, so
`DATASET` is one of `sparkov`, `tabformer`, `ieee_cis` (not `ulb`); the consistent attacker runs on `ieee_cis` and
`sparkov`. They take hours and write into `runs/<dataset>/attacks/` and `runs_consistent/<dataset>/attacks/`. Table V's
other inputs come from the attack grid and `fraudadv.hpo.fm_ablation`. Not run here (`RERUN = False`). The foundation
models need `chronos` imported before TensorFlow: set `RERUN = True` in a fresh kernel and run only the setup cell and
this one.

In [ ]:
RERUN = False
DATASET = "sparkov"

if RERUN:
    ALL = ["xgboost", "catboost", "mlp", "ft_transformer", "ivis", "hgnn", "hgt"]   # tabular and graph families
    FMS = ["chronos2", "timesfm"]
    import chronos  # noqa: F401  before TensorFlow (ivis), as history.run does for foundation models
    from fraudadv.attacks import history, history_consistent

    history.run(DATASET, ALL)                                   # history.csv, history_per_example.npz
    for f in FMS:
        history.run(DATASET, [f], tag=f"_fm_{f}")               # history_fm_<f>.csv
    if DATASET in ("ieee_cis", "sparkov"):                      # consistent attacker (runs_consistent/)
        history_consistent.run(DATASET, ALL)
        for f in FMS:
            history_consistent.run(DATASET, [f], tag=f"_fm_{f}")

## Check: every file this notebook wrote, against the reference build

In [ ]:
written = ["reports/table10_history.csv", "reports/table10_history.md", "reports/tables/foundation_models.csv",
           "reports/table12_groups.csv"]
if FM_ADAPT_RUNS:
    written += ["reports/table17_fm_adapt.csv", "reports/pareto_extra_fm_adapt.csv"]
nb.check(written)